# Exercise 1 - 3D - 2D

In [1]:
import cv2
import numpy as np

Recall from the slides the steps from Algorithm 3:

![title](algorithm_3.png)

![title](PnP.png)

# Exercise 1a)
The steps 1)-2.1) has already been done, and is saved in corresponding files. The exercise is to implement step 2.2) by filling in the missing code below

In [ ]:
# Why track points rather than re-detect them each frame: the whole pose-estimation
# approach that follows needs known 3D-2D correspondences (i.e., for each 3D
# landmark whose real-world position we already know, we need its current 2D
# pixel location in the new image. Optical flow tracking follows the same
# physical points from frame to frame, which is what preserves this point-to-point
# correspondence. Independently re-detecting features in each new frame would
# give a completely different set of points instead.
# Why calcOpticalFlowPyrLK() with these params: winSize/maxLevel control the
# pyramidal Lucas-Kanade search's neighborhood size and number of scale levels.
# Why reshape prev_points to (N, 1, 2) float32: this is the exact input
# shape/dtype cv2.calcOpticalFlowPyrLK() requires. The incoming 2D point array
# needs reshaping/casting to match that expected format before being passed in.
# Why filter by status == 1 (keeping only successfully tracked points): optical
# flow can fail to find some points in the new frame (status == 0). Any point
# that failed to track no longer has a valid 2D position, so it (and its
# corresponding 3D landmark) must be dropped from all three arrays together
# (world_points, prev_points, next_points) to keep every remaining index correctly
# aligned across all three. If you dropped a point from just one array, the
# indices would fall out of sync and later 3D-2D correspondences would be wrong.
def featureTracking(prev_img, next_img, prev_points, world_points):
    """
    Use OpenCV to find the prev_points from the prev_img in the next_img.
    Remember to remove points that could not be found from prev_points,
    next_points, and world_points.
    hint: status == 1
    """
    params = {
        "winSize": (21, 21),
        "maxLevel": 3,
        "criteria": (
            cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT,
            30,
            0.01,
        ),
    }

    # calcOpticalFlowPyrLK expects float32 points shaped (N, 1, 2).
    prev_points = np.asarray(prev_points, dtype=np.float32).reshape(-1, 1, 2)

    next_points, status, _ = cv2.calcOpticalFlowPyrLK(
        prev_img, next_img, prev_points, None, **params
    )

    # Keep only the points that were successfully tracked (status == 1), and
    # drop the corresponding entries from all three arrays so they stay in sync.
    status = status.reshape(-1).astype(bool)

    world_points = np.asarray(world_points)[status]
    prev_points = prev_points[status]
    next_points = next_points[status]

    return world_points, prev_points, next_points

Hint: Exercise 4 in week 2

# Exercise 1b)
Continue the algorithm by implementing step 2.3)

In [ ]:
# Run the full VO loop across 5 frames; tracking known 3D landmarks into each
# new frame, then solving for the camera's pose from those updated 3D-2D
# correspondences, one frame at a time.
# Why the camera matrix K is given: K (focal lengths + optical center) is an
# intrinsic camera property that doesn't change between frames.
# Why load pre-saved images/landmarks/2D references from .npy files:
# Steps 1)-2.1) of the algorithm (initial feature detection and initial 3D
# landmark triangulation) have already been done. This notebook picks up the
# pipeline starting from step 2.2) onward.
# Why call featureTracking() first, each iteration: this updates
# 'landmark_3D'/'reference_2D' to drop any points optical flow lost and produces
# 'tracked_2Dpoints'. The current frame's actual 2D positions of whichever 3D
# landmarks survived tracking, ready to feed into PnP.
# Why cv2.solvePnPRansac() rather than solvePnP(): given a set of 3D world
# points and their corresponding 2D image points, PnP ("Perspective-n-Point")
# solves for the camera's rotation/translation that would explain that projection.
# RANSAC is layered on top, making the solution robust to a small number of
# individually bad correspondences (e.g. from a slightly incorrect optical-flow
# track) rather than letting some outliers corrupt the whole pose estimate.
# Why invert (R, t) to get the camera's pose in the world frame: solvePnP's raw
# (rvec, tvec) output describes the transform that maps world points into the
# camera's own coordinate frame (X_cam = R@X_world + t), but what we actually
# want is the opposite: where the camera itself is, and how it's oriented,
# described in the world frame. Since R is a rotation matrix (and thus
# R^-1 = R^T, its transpose), the camera's world-frame position is -R^T @ t and
# its world-frame orientation is R^T; converted back to rvec via cv2.Rodrigues()
# for reporting together with tvec.
# Why cv2.Rodrigues() is used twice, in opposite directions: OpenCV represents
# rotations either as a 3-element rotation vector (rvec, an axis-angle
# representation) or as a full 3x3 rotation matrix. Rodrigues() converts between
# these two representations in either direction; here it first expands
# solvePnP's rvec output into a full matrix R (needed to compute R.T), then
# compresses the resulting world-frame rotation matrix R_world back down into an
# rvec for the final printed output.
# Why update reference_img = curImage at the end of each loop iteration: each
# new frame's landmarks need to be tracked starting from wherever they were
# actually located in the immediately preceding frame. After finishing this
# iteration's work, the "current" image becomes next iteration's "previous"
# reference image for the next round of optical flow tracking.
# Camera intrinsic matrix (or intrinsic calibration matrix).
K = np.array(
    [
        [7.188560e02, 0.000000e00, 6.071928e02],
        [0, 7.188560e02, 1.852157e02],
        [0, 0, 1],
    ]
)

reference_img = np.load("img_" + str(0) + ".npy")

cv2.setRNGSeed(0)
for t in range(1, 6):
    # The image at current time=t.
    curImage = np.load("img_" + str(t) + ".npy")
    # The 3D landmarks in the world coordinates which have been computed in
    # time=t-1.
    landmark_3D = np.load("landmark_" + str(t - 1) + ".npy")
    # The 2D coordinates of the 3D points in the previous frame at time=t-1.
    reference_2D = np.load("reference_2D_" + str(t - 1) + ".npy")

    # The 2D landmarks at the current time = t.
    landmark_3D, reference_2D, tracked_2Dpoints = featureTracking(
        reference_img, curImage, reference_2D, landmark_3D
    )

    """
    Using OpenCV, implement PnP using Ransac
    """
    _, rvec, tvec, inliers = cv2.solvePnPRansac(
        landmark_3D.astype(np.float64),
        tracked_2Dpoints.astype(np.float64),
        K.astype(np.float64),
        distCoeffs=None,
    )

    """
    Transform the translation and rotation into the world frame.
    """
    # solvePnP gives (rvec, tvec) that map world points into the camera frame:
    #   X_cam = R @ X_world + t
    # To express the camera's own pose in the world frame, we invert it:
    #   camera position in world = -R^T @ t
    #   camera orientation in world = R^T
    R, _ = cv2.Rodrigues(rvec)
    R_world = R.T
    tvec = -R.T @ tvec
    rvec, _ = cv2.Rodrigues(R_world)

    print(tvec[0], tvec[1], tvec[2], rvec[0], rvec[1], rvec[2])

    # Update for next timestep.
    reference_img = curImage

[-0.00110283] [-0.00067163] [-0.00078344] [7.40071674e-05] [7.35110576e-05] [-9.8454737e-05]
[-0.00363962] [-0.00875085] [0.67580836] [0.00216658] [-0.00325852] [0.00244333]
[-0.0109631] [-0.01635679] [1.3774088] [0.00364615] [-0.00751509] [0.00099692]
[-0.03156765] [-0.02560142] [2.09967816] [0.00509582] [-0.01121638] [0.00082977]
[-0.049719] [-0.03532475] [2.83300749] [0.00561427] [-0.01613329] [-0.00041981]


Hint: The output should look similar to:

[-0.00110282] [-0.00067164] [-0.00078343] [-7.40069212e-05] [-7.35119065e-05] [9.84544279e-05]

[-0.00363946] [-0.00875075] [0.67580842] [-0.0021666] [0.00325853] [-0.00244333]

[-0.01096271] [-0.01635663] [1.3774094] [-0.00364615] [0.0075151] [-0.00099691]

[-0.0315663] [-0.02560111] [2.0996797] [-0.00509583] [0.01121646] [-0.00082978]

[-0.04971858] [-0.03532535] [2.8330071] [-0.00561424] [0.0161333] [0.00041981]

# Exercise 1c)
What approximate direction did the camera move in?

Looking at the printed translations across the 5 frames, the x and y components stay very small while the z component grows steadily and dominates. From roughly 0 up to about 2.83 over the 5 steps, increasing by a consistent ~0.7 per frame. Since z is the camera's optical (forward-looking) axis, this means the camera moved almost entirely forward, along its own viewing direction (+Z), at roughly constant speed, with only tiny lateral (x) and vertical (y) drift. This matches the scenario the images come from: a forward-facing camera mounted on a vehicle driving straight ahead.